# HR Support Assistant Solution

This notebook creates a simple RAG-based HR support assistant using a ChromaDB knowledge base built from `data/support_ticket.txt`.

In [1]:
import asyncio
from pathlib import Path

import chromadb

try:
    import dotenv
    dotenv.load_dotenv()
except Exception:
    pass

try:
    from agents import Agent, Runner, function_tool, trace
except Exception:
    def function_tool(func):
        return func
    Agent = None
    Runner = None
    trace = None

repo_root = Path.cwd().resolve()
if repo_root.name == 'solutions':
    repo_root = repo_root.parent

print(f'Project root: {repo_root}')


Project root: /workspaces/Ai-Agent-course


In [2]:
chroma_client = chromadb.PersistentClient(str(repo_root / 'chroma'))
hr_support_db = chroma_client.get_collection(name='hr_support_qna')
print('HR support collection count:', hr_support_db.count())


HR support collection count: 213


In [3]:
@function_tool
def hr_support_lookup_tool(query: str, max_results: int = 3) -> str:
    """Search the HR support ticket knowledge base for relevant employee support answers."""
    results = hr_support_db.query(query_texts=[query], n_results=max_results)

    if not results['documents'][0]:
        return f'No HR support guidance found for: {query}'

    formatted_results = []
    for doc in results['documents'][0]:
        formatted_results.append(doc)

    return 'Relevant HR support answers:\n' + '\n\n'.join(formatted_results)


In [4]:
hr_support_agent = Agent(
    name='HR Support Assistant',
    instructions='''
    You are a helpful HR support assistant who responds to employee questions.
    Always use the hr_support_lookup_tool first when a question is related to leave,
    payroll, benefits, reimbursements, onboarding, policies, grievances, or employee support.
    Keep answers clear, practical, and concise.
    ''',
    tools=[hr_support_lookup_tool],
)
print('Agent ready:', hr_support_agent.name)


Agent ready: HR Support Assistant


In [5]:
async def run_example():
    if trace is not None:
        with trace('HR Support Assistant with HR support RAG'):
            result = await Runner.run(
                hr_support_agent,
                'How do I request annual leave from HR?',
            )
            print(result.final_output)
    else:
        print('The OpenAI Agents SDK is not installed in this environment. The notebook is set up, but the live agent runner is unavailable.')

await run_example()


Here are the steps to request annual leave:

1) Check leave balance and policy: confirm you have available annual leave and any notice requirements.  
2) Submit the request: use the HR portal or your manager approval workflow.  
   - Include: start date, end date, total days, and reason for leave (if required).  
3) Obtain approval: wait for your manager (and HR if applicable) to approve.  
4) Confirm and plan: once approved, update your calendar and notify teammates as needed.

If you’d like, I can draft a brief leave request email or guide you to the exact portal step if you share your company portal name.
